<a href="https://colab.research.google.com/github/Debdeep17092002/CODSOFT_TASKNO/blob/main/creditcardfrauddetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install numpy pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
PROJECT_DIR = Path('/content/drive/MyDrive/fraud_project')
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Create {PROJECT_DIR} in Google Drive and put fraudTrain.csv and fraudTest.csv there.')
os.chdir(PROJECT_DIR)
print('Project folder:', PROJECT_DIR)
print('Files:', sorted(p.name for p in PROJECT_DIR.iterdir()))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/fraud_project
Files: ['fraudTest.csv', 'fraudTrain.csv']


In [ ]:
from __future__ import annotations

import argparse
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd


CATEGORICAL = ["category", "state", "gender"]
NUMERIC = ["log_amount", "log_city_pop", "age", "distance_km", "hour_sin", "hour_cos", "weekday_sin", "weekday_cos", "month_sin", "month_cos"]


def read_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    required = {"trans_date_trans_time", "amt", "lat", "long", "merch_lat", "merch_long", "city_pop", "dob", "is_fraud"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")
    return df


def haversine_km(lat1: np.ndarray, lon1: np.ndarray, lat2: np.ndarray, lon2: np.ndarray) -> np.ndarray:
    rad = np.pi / 180.0
    dlat = (lat2 - lat1) * rad
    dlon = (lon2 - lon1) * rad
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1 * rad) * np.cos(lat2 * rad) * np.sin(dlon / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def make_features(df: pd.DataFrame, categories: dict[str, list[str]] | None = None):
    dt = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
    dob = pd.to_datetime(df["dob"], errors="coerce")
    hour = (dt.dt.hour + dt.dt.minute / 60).fillna(0).to_numpy()
    weekday = dt.dt.dayofweek.fillna(0).to_numpy()
    month = dt.dt.month.fillna(1).to_numpy()
    age = ((dt - dob).dt.days / 365.2425).fillna(0).to_numpy()
    numeric = np.column_stack([
        np.log1p(np.maximum(pd.to_numeric(df["amt"], errors="coerce").fillna(0).to_numpy(), 0)),
        np.log1p(np.maximum(pd.to_numeric(df["city_pop"], errors="coerce").fillna(0).to_numpy(), 0)),
        np.clip(age, 18, 100),
        haversine_km(*[pd.to_numeric(df[c], errors="coerce").fillna(0).to_numpy() for c in ("lat", "long", "merch_lat", "merch_long")]),
        np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24),
        np.sin(2 * np.pi * weekday / 7), np.cos(2 * np.pi * weekday / 7),
        np.sin(2 * np.pi * month / 12), np.cos(2 * np.pi * month / 12),
    ]).astype(np.float32)
    cats = {}
    fitted = {}
    for col in CATEGORICAL:
        values = df[col].fillna("<missing>").astype(str)
        vocab = categories[col] if categories else sorted(values.unique().tolist())
        codes = pd.Categorical(values, categories=vocab).codes
        onehot = np.zeros((len(df), len(vocab) + 1), dtype=np.float32)
        onehot[np.arange(len(df)), np.where(codes < 0, len(vocab), codes)] = 1
        cats[col] = onehot
        fitted[col] = vocab
    return np.column_stack([numeric, *(cats[k] for k in CATEGORICAL)]), fitted


def sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, -35, 35)
    return 1.0 / (1.0 + np.exp(-x))


class LogisticRegressionGD:
    def __init__(self, learning_rate: float = 0.12, iterations: int = 500, l2: float = 0.01):
        self.learning_rate, self.iterations, self.l2 = learning_rate, iterations, l2

    def fit(self, x: np.ndarray, y: np.ndarray):
        self.mean = x.mean(axis=0)
        self.scale = x.std(axis=0)
        self.scale[self.scale < 1e-6] = 1
        z = (x - self.mean) / self.scale
        self.w = np.zeros(z.shape[1] + 1, dtype=np.float64)
        design = np.column_stack([np.ones(len(z)), z])
        for i in range(self.iterations):
            p = sigmoid(design @ self.w)
            grad = design.T @ (p - y) / len(y)
            grad[1:] += self.l2 * self.w[1:]
            self.w -= self.learning_rate * grad
        return self

    def predict_proba(self, x: np.ndarray) -> np.ndarray:
        z = (x - self.mean) / self.scale
        return sigmoid(np.column_stack([np.ones(len(z)), z]) @ self.w)


class DecisionTree:
    def __init__(self, max_depth=6, min_leaf=40, max_features=None, bins=12, seed=0):
        self.max_depth, self.min_leaf, self.max_features, self.bins = max_depth, min_leaf, max_features, bins
        self.rng = np.random.default_rng(seed)

    @staticmethod
    def impurity(pos, total):
        p = pos / total
        return 2 * p * (1 - p)

    def _build(self, x, y, depth):
        positives = int(y.sum())
        node = {"p": positives / len(y)}
        if depth >= self.max_depth or len(y) < 2 * self.min_leaf or positives == 0 or positives == len(y):
            return node
        n_features = x.shape[1] if self.max_features is None else min(self.max_features, x.shape[1])
        features = self.rng.choice(x.shape[1], n_features, replace=False)
        parent = self.impurity(positives, len(y))
        best_gain, best = 0.0, None
        for f in features:
            col = x[:, f]
            cuts = np.unique(np.quantile(col, np.linspace(.1, .9, self.bins)))
            for cut in cuts:
                left = col <= cut
                n_left = int(left.sum())
                n_right = len(y) - n_left
                if n_left < self.min_leaf or n_right < self.min_leaf:
                    continue
                p_left = int(y[left].sum())
                p_right = positives - p_left
                gain = parent - (n_left * self.impurity(p_left, n_left) + n_right * self.impurity(p_right, n_right)) / len(y)
                if gain > best_gain:
                    best_gain, best = gain, (int(f), float(cut), left)
        if best is None:
            return node
        f, cut, left = best
        node.update(feature=f, cut=cut, left=self._build(x[left], y[left], depth + 1), right=self._build(x[~left], y[~left], depth + 1))
        return node

    def fit(self, x, y):
        self.root = self._build(x, y.astype(np.int8), 0)
        return self

    def predict_proba(self, x):
        out = np.empty(len(x), dtype=np.float32)
        stack = [(self.root, np.arange(len(x)))]
        while stack:
            node, rows = stack.pop()
            if "feature" not in node:
                out[rows] = node["p"]
                continue
            mask = x[rows, node["feature"]] <= node["cut"]
            stack.append((node["left"], rows[mask]))
            stack.append((node["right"], rows[~mask]))
        return out


class RandomForest:
    def __init__(self, n_trees=12, max_depth=6, min_leaf=40, seed=2026):
        self.n_trees, self.max_depth, self.min_leaf, self.seed = n_trees, max_depth, min_leaf, seed

    def fit(self, x, y):
        rng = np.random.default_rng(self.seed)
        positives = np.flatnonzero(y == 1)
        negatives = np.flatnonzero(y == 0)
        trees = []

        for i in range(self.n_trees):
            pos = rng.choice(positives, len(positives), replace=True)
            neg = rng.choice(negatives, len(positives), replace=True)
            idx = np.concatenate([pos, neg])
            rng.shuffle(idx)
            trees.append(DecisionTree(self.max_depth, self.min_leaf, max_features=max(1, int(math.sqrt(x.shape[1]))), seed=self.seed + i).fit(x[idx], y[idx]))
        self.trees = trees
        return self

    def predict_proba(self, x):
        return np.mean([tree.predict_proba(x) for tree in self.trees], axis=0)


def balanced_subset(indices: np.ndarray, y: np.ndarray, negatives_per_positive=8, seed=2026) -> np.ndarray:
    rng = np.random.default_rng(seed)
    pos = indices[y[indices] == 1]
    neg = indices[y[indices] == 0]
    take = min(len(neg), len(pos) * negatives_per_positive)
    result = np.concatenate([pos, rng.choice(neg, take, replace=False)])
    rng.shuffle(result)
    return result


def average_precision(y: np.ndarray, p: np.ndarray) -> float:
    order = np.argsort(-p, kind="stable")
    yy = y[order]
    tp = np.cumsum(yy == 1)
    fp = np.cumsum(yy == 0)
    recall = tp / max(1, int((y == 1).sum()))
    precision = tp / np.maximum(1, tp + fp)
    prev = np.concatenate([[0.0], recall[:-1]])
    return float(np.sum((recall - prev) * precision))


def threshold_for_f1(y, p):
    order = np.argsort(-p, kind="stable")
    yy, pp = y[order], p[order]
    tp = np.cumsum(yy == 1)
    fp = np.cumsum(yy == 0)
    fn = int((y == 1).sum()) - tp
    f1 = 2 * tp / np.maximum(1, 2 * tp + fp + fn)
    # Only consider thresholds after at least one predicted positive.
    i = int(np.argmax(f1))
    return float(pp[i]), float(f1[i])


def metrics(y, p, threshold):
    pred = p >= threshold
    tp = int(np.sum(pred & (y == 1)))
    fp = int(np.sum(pred & (y == 0)))
    fn = int(np.sum(~pred & (y == 1)))
    tn = int(np.sum(~pred & (y == 0)))
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return {"threshold": threshold, "precision": precision, "recall": recall, "f1": f1,
            "average_precision": average_precision(y, p), "true_positives": tp,
            "false_positives": fp, "false_negatives": fn, "true_negatives": tn}


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--train", default=r"D:\fraudTrain.csv")
    parser.add_argument("--test", default=r"D:\fraudTest.csv")
    parser.add_argument("--report", default="fraud_model_results.json")
    parser.add_argument("--model-output", default="fraud_tree_model.json")
    args = parser.parse_args()
    started = time.time()
    train, test = read_data(args.train), read_data(args.test)
    y_train = train["is_fraud"].astype(np.int8).to_numpy()
    y_test = test["is_fraud"].astype(np.int8).to_numpy()
    x_train, vocab = make_features(train)
    x_test, _ = make_features(test, vocab)

    split = int(len(train) * .8)
    fit_idx = np.arange(split)
    val_idx = np.arange(split, len(train))
    sample_idx = balanced_subset(fit_idx, y_train, negatives_per_positive=1)

    models = {
        "Logistic Regression": LogisticRegressionGD(iterations=120).fit(x_train[sample_idx], y_train[sample_idx]),
        "Decision Tree": DecisionTree(max_depth=6, min_leaf=60, seed=13).fit(x_train[sample_idx], y_train[sample_idx]),
        "Random Forest": RandomForest(n_trees=8, max_depth=5, min_leaf=60, seed=13).fit(x_train[sample_idx], y_train[sample_idx]),
    }
    results = {"data": {"train_rows": len(train), "train_fraud": int(y_train.sum()), "test_rows": len(test), "test_fraud": int(y_test.sum()), "features": x_train.shape[1]}, "models": {}}
    validation_f1 = {}
    for name, model in models.items():
        val_p = model.predict_proba(x_train[val_idx])
        threshold, val_f1 = threshold_for_f1(y_train[val_idx], val_p)
        validation_f1[name] = val_f1
        test_p = model.predict_proba(x_test)
        results["models"][name] = metrics(y_test, test_p, threshold)
        results["models"][name]["validation_f1"] = val_f1
        print(f"{name}: {json.dumps(results['models'][name])}", flush=True)
    best_name = max(validation_f1, key=validation_f1.get)
    selected = models[best_name]
    if not isinstance(selected, DecisionTree):
        raise TypeError("Only a Decision Tree can currently be exported for scoring")
    Path(args.model_output).write_text(json.dumps({
        "model": selected.__class__.__name__, "root": selected.root, "categories": vocab,
        "threshold": results["models"][best_name]["threshold"],
        "model_name": best_name, "feature_count": x_train.shape[1]
    }), encoding="utf-8")
    results["selected_model"] = best_name
    results["model_file"] = args.model_output
    results["elapsed_seconds"] = round(time.time() - started, 2)
    Path(args.report).write_text(json.dumps(results, indent=2), encoding="utf-8")
    print(f"Wrote {args.report}; elapsed {results['elapsed_seconds']}s")


if __name__ == "__main__":
    import sys
    sys.argv = [
        "fraud_model.py",
        "--train", str(PROJECT_DIR / "fraudTrain.csv"),
        "--test", str(PROJECT_DIR / "fraudTest.csv"),
        "--report", str(PROJECT_DIR / "fraud_model_results.json"),
        "--model-output", str(PROJECT_DIR / "fraud_tree_model.json"),
    ]
    main()


Logistic Regression: {"threshold": 0.8796598807654179, "precision": 0.3807415605976757, "recall": 0.32074592074592073, "f1": 0.3481781376518218, "average_precision": 0.22159497592267666, "true_positives": 688, "false_positives": 1119, "false_negatives": 1457, "true_negatives": 552455, "validation_f1": 0.41796322489391796}
Decision Tree: {"threshold": 1.0, "precision": 0.6529550827423168, "recall": 0.6438228438228438, "f1": 0.6483568075117371, "average_precision": 0.5211346489586743, "true_positives": 1381, "false_positives": 734, "false_negatives": 764, "true_negatives": 552840, "validation_f1": 0.6909340659340659}
Random Forest: {"threshold": 0.7451639175415039, "precision": 0.42120582120582123, "recall": 0.4722610722610723, "f1": 0.44527472527472534, "average_precision": 0.3540045163927193, "true_positives": 1013, "false_positives": 1392, "false_negatives": 1132, "true_negatives": 552182, "validation_f1": 0.5711382113821138}
Wrote /content/drive/MyDrive/fraud_project/fraud_model_resu